In [1]:
# =============================================================================
# CS-781 Phase 6 -- Abstractive legal summarization (BERTScore F1) with Arch #1 (decoder-only, Phase-1 model)
#
# Data: 7,030 (judgment, headnote) training pairs, 100 test judgments, no dev split (we hold out VAL_N training docs).
# Design
#   * One sequence per example:  <s> [document part] </s> [summary] </s>     (<= 1024 tokens = the Phase-1 context)
#       - the document header (appeal numbers, counsel names, "delivered by JUDGE") is stripped;
#       - DOC_MODE picks WHICH ~DOC_TOK tokens of the judgment the model reads: "head" (start of the body) or
#         "headtail" (first 2/3 + last 1/3)  -> the long-document analysis;
#       - the summary is cut to SUM_TOK = 1024 - 3 - DOC_TOK tokens (BERTScore only sees ~512 RoBERTa tokens anyway [assumption]).
#   * Loss only on the summary tokens (+ the closing </s> when the whole summary fits).
#   * Generation: greedy, batched, with no-repeat-ngram blocking, a repetition penalty and a minimum length.
#   * Local metrics on VAL docs: LM loss, BERTScore F1 (same call as the official evaluator), ROUGE-1/L, repetition,
#     and a length sweep (BERTScore of the first 100/200/300/400 words).
# Techniques (TECHNIQUES): full | lorma | lora | adapter | partial | normtune | kd   (KD = token-level logits distillation)
# Kaggle: paste each "# %%" block as its own cell, in order. FIRST RUN: TECHNIQUES=["lorma"] and read the "est. epoch time" line.
# NOT yet run on Kaggle -- the text/label/n-gram logic was tested on fake data; the torch code is untested.
# =============================================================================


# %% [1] Imports, HF login, metric packages
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, pickle, gc, traceback, functools, re, subprocess, sys
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

HAVE_METRICS = True
try:
    import bert_score, rouge_score
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score", "rouge-score"], capture_output=True, text=True)
    try:
        import bert_score, rouge_score
    except ImportError:
        HAVE_METRICS = False
        print("!! could not install bert-score / rouge-score (is Internet on?). Training/generation still work; local BERTScore is skipped.")
if HAVE_METRICS:
    from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| local metrics:", HAVE_METRICS)



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda | local metrics: True


In [2]:
# %% [2] Config
TECHNIQUES = ["full"]        # e.g. ["lorma"], ["adapter"], ["full"], ["kd"]  (run each in its own notebook if you like)
DOC_MODES = ["head"]          # head | headtail   (every technique runs once per mode)
FINAL_ALL = False             # True = train on ALL 7,030 pairs (no validation; last epoch is kept)
SEED = 42
SKIP_DONE = True
TEACHER_TECH = "full"         # KD only: the teacher = this technique's saved checkpoint
TEACHER_CKPT = ""             # KD only: path to the teacher's sum_best.pt ("" = working dir, else it retrains the teacher)

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/arch1final"                       # tokenizer (legal_bpe.json)
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"       # Arch #1 Phase-1 checkpoint

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {"train": "Phase_6_Summ/Train/train.json", "test": "Phase_6_Summ/Test/test.json"}

BLOCK = 3072                  # sequence length used for <s> doc </s> summary </s>. Arch #1 was pretrained with 1024.
ROPE_INTERP = True           # True + BLOCK > 1024: RoPE position interpolation (positions are scaled by 1024/BLOCK) so the model can read longer
                              #   inputs after fine-tuning (Chen et al. 2023). e.g. BLOCK=2048, DOC_TOK=1784 (summary budget stays 261), BATCH=2, ACCUM=4, GEN_BATCH=8
DOC_TOK = 2808                 # tokens of the judgment given to the model
SUM_TOK = BLOCK - 3 - DOC_TOK # tokens of the summary the model is trained on / allowed to generate
VAL_N = 300                   # training docs held out for validation (ignored when FINAL_ALL)
N_GEN_VAL = 60                # of those, how many are generated + scored with BERTScore after training
HEADER_SKIP_WORDS = 8         # words skipped after "delivered by" (judge name)

BATCH, ACCUM = 1, 8           # effective batch 8 sequences
WEIGHT_DECAY, WARMUP_FRAC, DROPOUT_FT = 0.01, 0.06, 0.1

# generation
GEN_BATCH = 4
MIN_GEN = 150                 # forbid </s> before this many generated tokens (references are long)
NO_REPEAT_NGRAM = 4
REP_PENALTY = 1.1
LEN_SWEEP_WORDS = [100, 150, 200, 250, 300, 400]
OUT_MAX_WORDS = None          # post-processing: cut every generated summary to ~this many words (at a sentence end if possible); None = keep as generated

LORMA_R, LORMA_ALPHA = 8, 8
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [0, 1, 3, 4, 6, 7, 9, 10, 11]   # KD student keeps 9 of 12 Phase-1 layers
KD_T, KD_ALPHA = 2.0, 0.5

HP = {   # starting points -- adjust after the epoch-time estimate and the first validation losses
    "full":     dict(epochs=3, lr=5e-5),
    "lorma":    dict(epochs=8, lr=4e-4),
    "lora":     dict(epochs=8, lr=4e-4),
    "adapter":  dict(epochs=8, lr=4e-4),
    "partial":  dict(epochs=6, lr=1e-4),
    "normtune": dict(epochs=10, lr=1e-3),
    "kd":       dict(epochs=4, lr=1e-4),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and TEACHER_TECH in TECHNIQUES:
    assert TECHNIQUES.index(TEACHER_TECH) < TECHNIQUES.index("kd"), "put the teacher technique before 'kd'"

ROOT_OUT = "/kaggle/working"
DOC_MODE = DOC_MODES[0]
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase6_arch1_{tech}_{DOC_MODE}{DOC_TOK}" + ("_pi" if ROPE_INTERP else "") + f"_s{SEED}" + ("_all" if FINAL_ALL else "")
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
print("Will run:", TECHNIQUES, "| doc modes:", DOC_MODES, f"| doc tokens {DOC_TOK} + summary tokens {SUM_TOK}", "| final-all:", FINAL_ALL)


Will run: ['full'] | doc modes: ['head'] | doc tokens 2808 + summary tokens 261 | final-all: False


In [3]:


# %% [3] Tokenizer (Phase-1 BPE), cached; documents are stored as uint16 arrays to save RAM
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
assert VOCAB_SIZE < 65536
EOW = "\u0100"

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

@functools.lru_cache(maxsize=None)
def encode_word_cached(word):
    return tuple(token_to_id.get(sym, UNK_ID) for sym in bpe_encode_word(word))

SPECIAL_IDS = {PAD_ID, UNK_ID, BOS_ID, EOS_ID}

def encode_words(words):
    ids = []
    for w in words:
        ids.extend(encode_word_cached(w))
    return ids

def decode_ids(ids):
    """token ids -> text (symbols are single bytes + the end-of-word marker)."""
    s = "".join(vocab[i] for i in ids if i not in SPECIAL_IDS and 0 <= i < len(vocab))
    out = []
    for w in s.split(EOW):
        if w:
            try:
                out.append(bytes(ord(c) for c in w).decode("utf-8", "ignore"))
            except ValueError:
                out.append("".join(c for c in w if ord(c) < 256))
    return " ".join(out)


# %% [4] Load Phase-6 data, strip the header, build the examples, print statistics
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

def read_records(path):
    raw = json.load(open(hf_local(path), encoding="utf-8"))
    if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], list):
        raw = raw["data"]
    if isinstance(raw, dict):
        out = []
        for k, v in raw.items():
            if isinstance(v, dict):
                d = dict(v); d.setdefault("id", k); out.append(d)
            else:
                out.append({"id": k, "document": v})
        return out
    return raw

def body_words(doc):
    """Judgment text without the header (appeal numbers, counsel) and the 'delivered by JUDGE' line."""
    text = " ".join(str(doc).split())
    i = text.find("delivered by")
    words = text[i:].split()[HEADER_SKIP_WORDS:] if i >= 0 else text.split()[100:]
    return words if len(words) >= 50 else text.split()

def doc_part(words, mode):
    """~DOC_TOK tokens of the judgment: 'head' = start of the body; 'headtail' = first 2/3 + last 1/3."""
    if mode == "head":
        return encode_words(words[: int(DOC_TOK * 1.4)])[:DOC_TOK]
    head_n, tail_n = DOC_TOK * 2 // 3, DOC_TOK - DOC_TOK * 2 // 3
    h = encode_words(words[: int(head_n * 1.4)])[:head_n]
    t = encode_words(words[-int(tail_n * 1.4):])[-tail_n:] if len(words) > int(head_n * 1.4) else []
    return h + t

def summary_part(summary):
    w = str(summary).split()
    ids = encode_words(w[: int(SUM_TOK * 1.3)])
    truncated = len(ids) > SUM_TOK or len(w) > int(SUM_TOK * 1.3)
    return ids[:SUM_TOK], truncated

def build_examples(recs, mode, with_summary=True):
    out = []
    for r in recs:
        prompt = [BOS_ID] + doc_part(body_words(r["document"]), mode) + [EOS_ID]
        ex = {"id": str(r["id"]), "prompt": prompt}
        if with_summary:
            s, trunc = summary_part(r["summary"])
            ex["sum"], ex["trunc"], ex["ref"] = s, trunc, str(r["summary"]).strip()
        out.append(ex)
    return out

t0 = time.time()
train_recs_all = read_records(FILES["train"]); test_recs = read_records(FILES["test"])
print(f"train {len(train_recs_all)} | test {len(test_recs)} | keys: {list(train_recs_all[0].keys())}")
assert len({str(r['id']) for r in test_recs}) == len(test_recs), "duplicate test ids"
rng_split = random.Random(SEED)
order = list(range(len(train_recs_all))); rng_split.shuffle(order)
val_idx = set() if FINAL_ALL else set(order[:VAL_N])
val_recs = [train_recs_all[i] for i in sorted(val_idx)]
fit_recs = [r for i, r in enumerate(train_recs_all) if i not in val_idx]
print(f"fit docs {len(fit_recs)} | validation docs {len(val_recs)} | tokenising ...")

EX = {}   # per DOC_MODE caches
def get_examples(mode):
    if mode not in EX:
        EX[mode] = {"fit": build_examples(fit_recs, mode), "val": build_examples(val_recs, mode), "test": build_examples(test_recs, mode, False)}
    return EX[mode]
_e = get_examples(DOC_MODES[0])
print(f"tokenised in {time.time()-t0:.0f}s")
pl = np.array([len(e["prompt"]) for e in _e["fit"]]); sl = np.array([len(e["sum"]) for e in _e["fit"]])
print(f"prompt tokens: median {int(np.median(pl))} min {pl.min()} | prompts shorter than the budget: {(pl < DOC_TOK + 2).mean():.1%}")
print(f"summary tokens kept: median {int(np.median(sl))} | truncated summaries: {np.mean([e['trunc'] for e in _e['fit']]):.1%}")
print("body words example:", " ".join(body_words(train_recs_all[0]['document'])[:30]))
print("test prompt lengths:", dict(Counter(len(e['prompt']) for e in _e['test']).most_common(4)))


# %% [5] Dataset: <s> doc </s> summary [</s>], loss only on the summary part
class SumDataset(Dataset):
    def __init__(self, examples):
        self.ex = examples
    def __len__(self):
        return len(self.ex)
    def __getitem__(self, i):
        e = self.ex[i]
        seq = e["prompt"] + e["sum"] + ([] if e["trunc"] else [EOS_ID])
        sep = len(e["prompt"]) - 1                       # position of the </s> after the document
        return seq, sep

def make_labels(seq, sep):
    """labels[t] = seq[t+1] for t in [sep, len-2]  (the model predicts the summary after the separator), else -100."""
    lab = [-100] * len(seq)
    for t in range(sep, len(seq) - 1):
        lab[t] = seq[t + 1]
    return lab

def collate(batch):
    T = max(len(s) for s, _ in batch)
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), T), -100, dtype=torch.long)
    for i, (s, sep) in enumerate(batch):
        ids[i, :len(s)] = torch.tensor(s)
        lab[i, :len(s)] = torch.tensor(make_labels(s, sep))
    return ids, lab

def make_loader(examples, shuffle):
    return DataLoader(SumDataset(examples), batch_size=BATCH, shuffle=shuffle, collate_fn=collate, num_workers=0)


# %% [6] Model: Arch #1 backbone + PEFT modules (LM head tied to the embeddings)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0, scale=1.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float() * scale      # scale < 1 = position interpolation
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """h' = h + (alpha/r) B(A h) = ((alpha/r) B A + I) W0 x with h = W0 x (multiplicative; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class LoRAAdapter(nn.Module):
    """Additive update (alpha/r) B(A x), added to W0 x (LoRA; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * F.linear(F.linear(x, self.A), self.B)

class BottleneckAdapter(nn.Module):
    """Houlsby-style adapter: h + up(gelu(down(h))); `up` starts at zero (identity at init)."""
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.adapt_mode = None
        self.q_ad = None
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        if self.adapt_mode == "lorma":
            q, v = self.q_ad(q), self.v_ad(v)
        elif self.adapt_mode == "lora":
            q, v = q + self.q_ad(x), v + self.v_ad(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
        self.ad1 = None
        self.ad2 = None
    def forward(self, x, cos, sin):
        a = self.attn(self.ln1(x), cos, sin)
        if self.ad1 is not None:
            a = self.ad1(a)
        x = x + a
        f = self.ffn(self.ln2(x))
        if self.ad2 is not None:
            f = self.ad2(f)
        return x + f

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout, rope_scale=1.0):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu", scale=rope_scale)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
assert BLOCK <= cfg["block_size"] or ROPE_INTERP, f"BLOCK {BLOCK} exceeds the model context {cfg['block_size']}: set ROPE_INTERP=True"
ROPE_SCALE = cfg["block_size"] / BLOCK if (ROPE_INTERP and BLOCK > cfg["block_size"]) else 1.0
print(f"sequence length {BLOCK} | RoPE position scale {ROPE_SCALE}")
print(f"Phase-1 model: {cfg['n_layer']} layers, d={cfg['n_embd']}, context {cfg['block_size']}")

def make_student_state(sd, keep):
    new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split(".")
            old = int(parts[1])
            if old in keep:
                parts[1] = str(keep.index(old))
                new[".".join(parts)] = v
        else:
            new[k] = v
    return new

def build_model(technique, keep_layers=None):
    sd, n_layer = ckpt["model_state_dict"], cfg["n_layer"]
    if keep_layers is not None:
        sd, n_layer = make_student_state(sd, keep_layers), len(keep_layers)
    bb = LegalLLaMA(cfg["vocab_size"], n_layer, cfg["n_head"], cfg["n_embd"],
                    max(BLOCK, cfg["block_size"]), cfg["ffn_mult"], DROPOUT_FT, rope_scale=ROPE_SCALE)
    bb.load_state_dict(sd)
    d = cfg["n_embd"]
    if technique in ("lorma", "lora", "adapter", "partial", "normtune"):
        for p in bb.parameters():
            p.requires_grad = False
        if technique in ("lorma", "lora"):
            for blk in bb.blocks:
                blk.attn.adapt_mode = technique
                make = LoRMAPlus if technique == "lorma" else LoRAAdapter
                blk.attn.q_ad, blk.attn.v_ad = make(d, LORMA_R, LORMA_ALPHA), make(d, LORMA_R, LORMA_ALPHA)
        elif technique == "adapter":
            for blk in bb.blocks:
                blk.ad1, blk.ad2 = BottleneckAdapter(d, ADAPTER_DIM), BottleneckAdapter(d, ADAPTER_DIM)
        elif technique == "partial":
            for blk in bb.blocks[-PARTIAL_LAYERS:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in bb.ln_f.parameters():
                p.requires_grad = True
        elif technique == "normtune":
            for m in bb.modules():
                if isinstance(m, RMSNorm):
                    m.weight.requires_grad = True
    bb = bb.to(device)
    n_tot = sum(p.numel() for p in bb.parameters())
    n_trn = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    print(f"[{technique}{' student' if keep_layers else ''}] trainable {n_trn/1e6:.3f}M of {n_tot/1e6:.1f}M params ({100*n_trn/n_tot:.2f}%)")
    return bb


# %% [7] Generation: batched greedy decoding with no-repeat-ngram blocking, repetition penalty and a minimum length
@torch.no_grad()
def gen_batch(net, ids, max_new, min_new):
    B = ids.size(0); n = NO_REPEAT_NGRAM
    gen = [[] for _ in range(B)]; done = [False] * B; banned = [dict() for _ in range(B)]
    cur = ids
    for step in range(max_new):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            h = net.forward_hidden(cur)[:, -1]
            logits = net.head(h).float()
        if REP_PENALTY != 1.0:
            for b in range(B):
                if gen[b]:
                    t = torch.tensor(sorted(set(gen[b])), device=logits.device)
                    l = logits[b, t]
                    logits[b, t] = torch.where(l > 0, l / REP_PENALTY, l * REP_PENALTY)
        logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e9
        if step < min_new:
            logits[:, EOS_ID] = -1e9
        if n > 1:
            for b in range(B):
                if len(gen[b]) >= n - 1:
                    ban = banned[b].get(tuple(gen[b][-(n - 1):]))
                    if ban:
                        logits[b, list(ban)] = -1e9
        nxt = logits.argmax(-1)
        for b in range(B):
            if done[b]:
                continue
            t = int(nxt[b]); gen[b].append(t)
            if t == EOS_ID:
                done[b] = True
            elif n > 1 and len(gen[b]) >= n:
                banned[b].setdefault(tuple(gen[b][-n:-1]), set()).add(t)
        if all(done):
            break
        cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
    return gen

def generate_all(net, examples, tag=""):
    """Groups prompts of equal length (almost all prompts are exactly DOC_TOK+2 long) so no padding is needed."""
    net.eval()
    groups = {}
    for i, e in enumerate(examples):
        groups.setdefault(len(e["prompt"]), []).append(i)
    outs, t0, done_n = [None] * len(examples), time.time(), 0
    for L, idxs in groups.items():
        for s in range(0, len(idxs), GEN_BATCH):
            b = idxs[s:s + GEN_BATCH]
            ids = torch.tensor([examples[i]["prompt"] for i in b], device=device)
            res = gen_batch(net, ids, min(SUM_TOK, BLOCK - L), MIN_GEN)
            for i, g in zip(b, res):
                outs[i] = g
            done_n += len(b)
            print(f"  [{tag}] generated {done_n}/{len(examples)} ({time.time()-t0:.0f}s)")
    texts = []
    for e, g in zip(examples, outs):
        txt = decode_ids(g).strip()
        texts.append(txt if txt else " ".join(e.get("fallback", ["No summary available."])))
    return texts, outs


# %% [8] Evaluation helpers: validation LM loss, BERTScore / ROUGE, repetition statistics
@torch.no_grad()
def val_loss(net, examples):
    net.eval()
    tot, cnt = 0.0, 0
    for ids, lab in make_loader(examples, False):
        ids, lab = ids.to(device), lab.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            h = net.forward_hidden(ids)
            m = lab != -100
            logits = net.head(h[m]).float()
        tot += F.cross_entropy(logits, lab[m], reduction="sum").item(); cnt += int(m.sum())
    return tot / max(cnt, 1)

def distinct_ngram_ratio(texts, n=4):
    seen, total = set(), 0
    for t in texts:
        w = t.split()
        for i in range(len(w) - n + 1):
            seen.add(tuple(w[i:i + n])); total += 1
    return len(seen) / max(total, 1)

_BS = None

def trunc_text(text, n):
    """First n words, ended at the last sentence end inside the final 30% when there is one."""
    w = text.split()
    if n is None or len(w) <= n:
        return text
    cut = " ".join(w[:n])
    k = cut.rfind(". ")
    return cut[: k + 1] if k > 0.7 * len(cut) else cut

def metrics_vs_refs(preds, refs):
    out = {}
    if not HAVE_METRICS:
        return out
    global _BS
    if _BS is None:
        _BS = bert_score.BERTScorer(lang="en", device=device)        # roberta-large, same defaults as the official evaluator's call
    P, R, F1 = _BS.score(preds, refs, batch_size=8)
    sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1 = np.mean([sc.score(rf, pr)["rouge1"].fmeasure for pr, rf in zip(preds, refs)])
    rl = np.mean([sc.score(rf, pr)["rougeL"].fmeasure for pr, rf in zip(preds, refs)])
    out.update(bertscore_f1=float(F1.mean()), bertscore_p=float(P.mean()), bertscore_r=float(R.mean()), rouge1=float(r1), rougeL=float(rl))
    return out

def full_eval(preds, refs):
    out = metrics_vs_refs(preds, refs)
    out["avg_words"] = float(np.mean([len(p.split()) for p in preds]))
    out["avg_ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    out["distinct_4gram"] = distinct_ngram_ratio(preds)
    if HAVE_METRICS:
        out["length_sweep"] = {}
        for w in LEN_SWEEP_WORDS:
            out["length_sweep"][w] = metrics_vs_refs([" ".join(p.split()[:w]) or "x" for p in preds], refs).get("bertscore_f1")
    return out


# %% [9] Training function (single learning rate; optional teacher for token-level distillation)
def fit(model, examples, epochs, lr, best_path, teacher=None, val_examples=None):
    loader = make_loader(examples, True)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    steps_per_epoch = math.ceil(len(loader) / ACCUM)
    total_steps = epochs * steps_per_epoch
    warm = max(1, int(WARMUP_FRAC * total_steps))
    def lr_factor(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))
    best, step, t0, history, eta_done = 1e9, 0, time.time(), [], False
    if teacher is not None:
        teacher.eval()
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n_b = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for bi, (ids, lab) in enumerate(loader):
            ids, lab = ids.to(device), lab.to(device)
            m = lab != -100
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model.head(model.forward_hidden(ids)[m])
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher.head(teacher.forward_hidden(ids)[m])
            logits = logits.float()
            loss = F.cross_entropy(logits, lab[m])
            if teacher is not None:
                kd = F.kl_div(F.log_softmax(logits / KD_T, dim=-1), F.softmax(t_logits.float() / KD_T, dim=-1),
                              reduction="batchmean") * (KD_T ** 2)
                loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd
            scaler.scale(loss / ACCUM).backward()
            run_loss += loss.item(); n_b += 1
            if (bi + 1) % ACCUM == 0 or bi + 1 == len(loader):
                for g in optimizer.param_groups:
                    g["lr"] = lr * lr_factor(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1
            if not eta_done and bi == 49:
                per = (time.time() - t0) / 50
                print(f"  est. epoch time {per*len(loader)/60:.1f} min ({len(loader)} batches, {per:.2f}s/batch) "
                      f"-> {epochs} epochs ~ {epochs*per*len(loader)/60:.0f} min (+ validation)")
                eta_done = True
        rec = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
        if val_examples:
            vl = val_loss(model, val_examples); rec["val_loss"] = vl
            print(f"[epoch {epoch}] train loss {rec['train_loss']:.4f} | val LM loss {vl:.4f} (ppl {math.exp(vl):.1f}) | {(time.time()-t0)/60:.1f}m")
            if vl < best:
                best = vl
                torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": vl}, best_path)
                print("  -> new best checkpoint saved")
        else:
            print(f"[epoch {epoch}] train loss {rec['train_loss']:.4f} (all-data run; no validation) | {(time.time()-t0)/60:.1f}m")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": None}, best_path)
        history.append(rec)
    model.load_state_dict(torch.load(best_path, map_location="cpu")["model"])
    return history


# %% [10] One technique end-to-end (train -> validation generation + metrics -> test generation -> files)
def set_seed(s):
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def run_technique(tech):
    out_dir = out_dir_for(tech)
    os.makedirs(out_dir, exist_ok=True)
    best_path = os.path.join(out_dir, "sum_best.pt")
    pred_path = os.path.join(out_dir, "prediction.json")
    summ_path = os.path.join(out_dir, "summary.json")
    if SKIP_DONE and all(os.path.exists(p) for p in [pred_path, summ_path]):
        print(f"\n##### [{tech}/{DOC_MODE}] already finished -> skipping")
        return json.load(open(summ_path))
    print(f"\n{'='*78}\n##### Phase 6 | Arch1 | {tech} | doc mode {DOC_MODE} ({DOC_TOK} doc + {SUM_TOK} summary tokens) | seed {SEED} | {HP[tech]}\n{'='*78}")
    set_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    ex = get_examples(DOC_MODE)
    val_ex = ex["val"] if val_recs else None
    t_run = time.time()
    teacher = None
    if tech == "kd":
        teacher = build_model(TEACHER_TECH)
        t_ckpt = TEACHER_CKPT or os.path.join(out_dir_for(TEACHER_TECH), "sum_best.pt")
        if os.path.exists(t_ckpt):
            teacher.load_state_dict(torch.load(t_ckpt, map_location="cpu")["model"])
            print("Loaded teacher from", t_ckpt)
        else:
            print(f"No saved teacher found -> training the teacher ({TEACHER_TECH}) first")
            fit(teacher, ex["fit"], best_path=os.path.join(out_dir, "teacher_best.pt"), val_examples=val_ex, **HP[TEACHER_TECH])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        if val_ex:
            print(f"TEACHER val LM loss: {val_loss(teacher, val_ex):.4f}")
        model = build_model("full", keep_layers=STUDENT_KEEP)
        t_p, s_p = sum(p.numel() for p in teacher.parameters()), sum(p.numel() for p in model.parameters())
        print(f"Teacher {t_p/1e6:.1f}M -> student {s_p/1e6:.1f}M params ({100*(1-s_p/t_p):.0f}% smaller)")
        history = fit(model, ex["fit"], best_path=best_path, teacher=teacher, val_examples=val_ex, **HP["kd"])
    else:
        model = build_model(tech)
        history = fit(model, ex["fit"], best_path=best_path, val_examples=val_ex, **HP[tech])
    json.dump(history, open(os.path.join(out_dir, "history.json"), "w"), indent=2)
    train_min = (time.time() - t_run) / 60
    print(f"[{tech}] training finished in {train_min:.1f} min")

    result = {"technique": tech, "doc_mode": DOC_MODE, "doc_tok": DOC_TOK, "sum_tok": SUM_TOK, "train_minutes": round(train_min, 1)}
    if val_ex:
        result["val_loss"] = val_loss(model, val_ex)
        nv = min(N_GEN_VAL, len(val_ex))
        t_g = time.time()
        vtexts, _ = generate_all(model, val_ex[:nv], "val")
        vtexts = [trunc_text(t, OUT_MAX_WORDS) for t in vtexts]
        result["val_gen_sec"] = round(time.time() - t_g, 1)
        refs = [e["ref"] for e in val_ex[:nv]]
        ev = full_eval(vtexts, refs)
        result["val_metrics"] = ev
        json.dump([{"id": e["id"], "pred": p, "ref": r} for e, p, r in zip(val_ex[:nv], vtexts, refs)],
                  open(os.path.join(out_dir, "val_generations.json"), "w"))
        print("VALIDATION:", json.dumps(ev, indent=1))
        print("EXAMPLE PRED:", vtexts[0][:600]); print("EXAMPLE REF :", refs[0][:600])

    test_ex = ex["test"]
    for e, r in zip(test_ex, test_recs):
        e["fallback"] = body_words(r["document"])[:100] or ["No summary available."]
    t_i = time.time()
    ttexts, _ = generate_all(model, test_ex, "test")
    ttexts = [trunc_text(t, OUT_MAX_WORDS) for t in ttexts]
    result["test_gen_sec"] = round(time.time() - t_i, 1)
    pred = [{"id": e["id"], "summary": t} for e, t in zip(test_ex, ttexts)]
    assert len(pred) == len(test_recs) and all(p["summary"].strip() for p in pred)
    assert {p["id"] for p in pred} == {str(r["id"]) for r in test_recs}
    json.dump(pred, open(pred_path, "w"))
    print(f"[{tech}] test summaries {len(pred)} | avg words {np.mean([len(p['summary'].split()) for p in pred]):.0f} "
          f"| distinct-4gram {distinct_ngram_ratio([p['summary'] for p in pred]):.2f} | generation {result['test_gen_sec']}s")
    n_tot = sum(p.numel() for p in model.parameters()); n_trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    result.update(params_total_M=round(n_tot / 1e6, 2), params_trainable_M=round(n_trn / 1e6, 3),
                  peak_gpu_mem_GB=round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None)
    json.dump(result, open(summ_path, "w"), indent=2)
    del model, teacher
    return result


# %% [11] Run everything chosen in cell [2]
all_results = []
for _mode in DOC_MODES:
    DOC_MODE = _mode
    for tech in TECHNIQUES:
        try:
            res = run_technique(tech)
        except Exception:
            print(f"\n!!!!! [{tech}/{DOC_MODE}] FAILED -- continuing")
            traceback.print_exc()
            res = {"technique": tech, "doc_mode": DOC_MODE, "error": True}
        all_results.append(res)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print("\n" + "=" * 78 + "\nSUMMARY\n" + "=" * 78)
for r in all_results:
    if r.get("error"):
        print(f"{r['technique']:9s} {r['doc_mode']:9s} FAILED"); continue
    vm = r.get("val_metrics", {})
    print(f"{r['technique']:9s} {r['doc_mode']:9s} val loss {r.get('val_loss', float('nan')):.3f} | BERTScore F1 {vm.get('bertscore_f1', float('nan')):.4f} "
          f"| R1 {vm.get('rouge1', float('nan')):.3f} RL {vm.get('rougeL', float('nan')):.3f} | train {r['train_minutes']} min | "
          f"params {r['params_total_M']}M (trainable {r['params_trainable_M']}M) | mem {r['peak_gpu_mem_GB']} GB")


# %% [12] Zips: one Codabench submission zip per run (prediction.json at the zip root) + a results zip
SUB_DIR, RES_DIR = f"{ROOT_OUT}/submissions", f"{ROOT_OUT}/results_zips"
os.makedirs(SUB_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)
for _mode in DOC_MODES:
    DOC_MODE = _mode
    for t in TECHNIQUES:
        d = out_dir_for(t)
        if not os.path.exists(f"{d}/prediction.json"):
            print(f"[{t}/{_mode}] prediction.json not found -> skipped"); continue
        tag = f"phase6_arch1_{t}_{_mode}{DOC_TOK}" + ("_pi" if ROPE_INTERP else "") + f"_s{SEED}" + ("_all" if FINAL_ALL else "")
        with zipfile.ZipFile(f"{SUB_DIR}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(f"{d}/prediction.json", arcname="prediction.json")
        with zipfile.ZipFile(f"{RES_DIR}/results_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            for fn in ["prediction.json", "val_generations.json", "history.json", "summary.json"]:
                if os.path.exists(f"{d}/{fn}"):
                    zf.write(f"{d}/{fn}", arcname=fn)
        print(f"[{t}/{_mode}] submission_{tag}.zip")
print("Upload the zips in", SUB_DIR, "to Codabench.")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

train 7030 | test 100 | keys: ['id', 'document', 'summary']
fit docs 6730 | validation docs 300 | tokenising ...
tokenised in 20s
prompt tokens: median 2810 min 93 | prompts shorter than the budget: 41.0%
summary tokens kept: median 261 | truncated summaries: 94.4%
body words example: by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal No.768/77 relates
test prompt lengths: {2810: 61, 1371: 1, 2471: 1, 2555: 1}
sequence length 3072 | RoPE position scale 0.3333333333333333
Phase-1 model: 12 layers, d=576, context 1024

##### Phase 6 | Arch1 | full | doc mode head (2808 doc + 261 summary tokens) | seed 42 | {'epochs': 3, 'lr': 5e-05}
[full] trainable 66.222M of 66.2M params (100.00%)
  est. epoch time 21.3 min (6730 batches, 0.19s/batch) -> 3 epochs ~ 64 min (+ validation)
[epoch 1] train loss 3.0016 | val LM loss 2.2158 (ppl 9.2) | 21.3m
  -> new best checkpoint saved
[ep

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


VALIDATION: {
 "bertscore_f1": 0.8450300693511963,
 "bertscore_p": 0.8573715686798096,
 "bertscore_r": 0.8332312107086182,
 "rouge1": 0.38516898263412,
 "rougeL": 0.2129794323323982,
 "avg_words": 240.81666666666666,
 "avg_ref_words": 875.4833333333333,
 "distinct_4gram": 0.9051089775036794,
 "length_sweep": {
  "100": 0.8352413177490234,
  "150": 0.8410815596580505,
  "200": 0.8437631130218506,
  "250": 0.8449903130531311,
  "300": 0.8450300693511963,
  "400": 0.8450300693511963
 }
}
EXAMPLE PRED: The appellant was a mortgagee in possession of certain properties and his interest in the said property was transferred to the mortgagee under the provisions of the Evacuee Property (Central) Rules, 1950 and he was entitled to obtain possession of the said property. The appellant made an application to the Competent Officer under section 7 of the Act for obtaining possession of the mortgaged property against payment of the mortgage debt and the Custodian directed the appellant to hand over v